# ADNI ConvNeXt

Load the grouped ADNI samples and inspect their MRI slices before training.

In [ ]:
import math

import matplotlib.pyplot as plt
import torch

from dataset import create_adni_dataloaders

print(f"PyTorch {torch.__version__}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

In [ ]:
SEED = 42
BATCH_SIZE = 8
IMAGE_SIZE = (224, 224)

train_loader, validation_loader, test_loader = create_adni_dataloaders(
    batch_size=BATCH_SIZE,
    seed=SEED,
    image_size=IMAGE_SIZE,
)

print(f"Training samples:   {len(train_loader.dataset)}")
print(f"Validation samples: {len(validation_loader.dataset)}")
print(f"Test samples:       {len(test_loader.dataset)}")
print(f"Classes: {train_loader.dataset.dataset.class_to_idx}")

In [ ]:
batch = next(iter(train_loader))

print(f"Images: {tuple(batch['images'].shape)}")
print(f"Labels: {tuple(batch['labels'].shape)}")
print(f"Sample IDs: {batch['sample_ids']}")
print(f"Slice counts: {batch['slice_mask'].sum(dim=1).tolist()}")

In [ ]:
sample_index = 0
valid_slices = batch['slice_mask'][sample_index]
images = batch['images'][sample_index][valid_slices]
slice_indices = batch['slice_indices'][sample_index]

columns = min(8, len(images))
rows = math.ceil(len(images) / columns)
figure, axes = plt.subplots(rows, columns, figsize=(2 * columns, 2 * rows), squeeze=False)

for axis, image, slice_index in zip(axes.flat, images, slice_indices):
    axis.imshow(image.permute(1, 2, 0).cpu())
    axis.set_title(f"Slice {slice_index}")
    axis.axis('off')

for axis in list(axes.flat)[len(images):]:
    axis.axis('off')

figure.suptitle(f"Sample: {batch['sample_ids'][sample_index]}")
figure.tight_layout()